# 01 — Transfer learning with ResNet50 (PyTorch)

PyTorch-only lesson using torchvision pretrained weights. Dataset: CIFAR-10 for a self-contained example; adapt the folder dataset for Homer/Bart.

## 1. Environment and pretrained preprocessing

In [ ]:
import torch, torchvision
from torchvision import datasets,transforms
from torchvision.models import resnet50,ResNet50_Weights
from torch.utils.data import DataLoader
from torch import nn
print(torch.__version__,torchvision.__version__)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights=ResNet50_Weights.DEFAULT; preprocess=weights.transforms()
train_tf=transforms.Compose([transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
test_tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
train=datasets.CIFAR10('data',train=True,download=True,transform=train_tf); test=datasets.CIFAR10('data',train=False,download=True,transform=test_tf)
train_loader=DataLoader(train,batch_size=32,shuffle=True,num_workers=2); test_loader=DataLoader(test,batch_size=64)

## 2. Freeze backbone and replace classifier

In [ ]:
model=resnet50(weights=weights)
for p in model.parameters(): p.requires_grad=False
model.fc=nn.Linear(model.fc.in_features,10)
model=model.to(device)
optimizer=torch.optim.AdamW(model.fc.parameters(),lr=1e-3); criterion=nn.CrossEntropyLoss()
print('Trainable params:',sum(p.numel() for p in model.parameters() if p.requires_grad))

## 3. Train classifier head

In [ ]:
for epoch in range(3):
 model.train(); total=0
 for x,y in train_loader:
  x,y=x.to(device),y.to(device); optimizer.zero_grad(set_to_none=True); loss=criterion(model(x),y); loss.backward(); optimizer.step(); total+=loss.item()*len(y)
 print('epoch',epoch+1,'loss',total/len(train))

## 4. Fine-tune layer4 with smaller learning rate

In [ ]:
for name,p in model.named_parameters():
 if name.startswith('layer4.'): p.requires_grad=True
optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-5,weight_decay=1e-4)
for epoch in range(2):
 model.train()
 for x,y in train_loader:
  x,y=x.to(device),y.to(device); optimizer.zero_grad(set_to_none=True); loss=criterion(model(x),y); loss.backward(); optimizer.step()
 print('fine-tuning epoch',epoch+1)

## 5. Test evaluation and saving

In [ ]:
from sklearn.metrics import classification_report
model.eval(); ys=[]; ps=[]
with torch.inference_mode():
 for x,y in test_loader:
  p=model(x.to(device)).argmax(1).cpu(); ys.extend(y.tolist()); ps.extend(p.tolist())
print(classification_report(ys,ps,target_names=train.classes))
torch.save({'state_dict':model.state_dict(),'classes':train.classes},'resnet50_cifar10.pt')

## Tips

Use the preprocessing bundled with the selected weights. Compare frozen vs fine-tuned models with identical splits. Do not claim results until the full experiment has run.